In [100]:
import pandas as pd
windows = pd.read_pickle('/old/old_analysis_cache/Zombies_response_windows.pkl')

In [101]:
sig = pd.read_pickle('/old/old_analysis_cache/Zombies_significant_windows_pANOVAorGLM_passed.pkl')
sig

,NeuronID,GLM_p-value,WindowStart_ms,WindowEnd_ms,F-statistic,PermANOVA_p-value,GLM_pval_corrected,GLM_significant,Permutation_pval_corrected,Permutation_significant
0,AMG_2023-09-26_1_Channel.C_014_Unit 1,0.000000,150,500,1.640527,0.123,0.000000,True,0.840000,False
1,AMG_2023-09-26_1_Channel.C_018_Unit 1,0.028004,50,150,0.875831,0.546,0.036111,True,0.897816,False
2,AMG_2023-09-26_1_Channel.C_018_Unit 1,0.028004,1750,1850,0.850747,0.569,0.036111,True,0.897816,False
3,AMG_2023-09-26_1_Channel.C_018_Unit 2,0.010077,100,200,0.867503,0.529,0.016187,True,0.897816,False
4,AMG_2023-09-26_1_Channel.C_018_Unit 2,0.010077,350,450,1.542621,0.157,0.016187,True,0.862457,False
...,...,...,...,...,...,...,...,...,...,...
730,Unknown_2023-12-18_3_Channel.C_015_Unit 1,0.011281,1300,1400,1.030063,0.435,0.017870,True,0.897816,False
731,Unknown_2023-12-18_3_Channel.C_015_Unit 1,0.011281,2000,2150,2.180261,0.029,0.017870,True,0.645909,False
732,Unknown_2023-12-18_3_Channel.C_021_Unit 1,0.010047,600,800,1.065941,0.358,0.016187,True,0.897816,False
733,Unknown_2023-12-18_3_Channel.C_021_Unit 1,0.010047,1000,1300,1.104037,0.373,0.016187,True,0.897816,False


In [102]:
my_list = sig[sig['PermANOVA_p-value']<0.05]
my_list

,NeuronID,GLM_p-value,WindowStart_ms,WindowEnd_ms,F-statistic,PermANOVA_p-value,GLM_pval_corrected,GLM_significant,Permutation_pval_corrected,Permutation_significant
29,AMG_2023-09-26_3_Channel.C_027_Unit 2,4.677418e-83,150,350,2.101942,0.043,6.139111e-82,True,0.750978,False
54,AMG_2023-10-03_3_Channel.C_006_Unit 1,7.350817e-04,200,400,4.400109,0.001,1.530553e-03,True,0.066818,False
56,AMG_2023-10-03_3_Channel.C_006_Unit 1,7.350817e-04,2150,2250,4.370457,0.000,1.530553e-03,True,0.000000,True
83,AMG_2023-10-03_4_Channel.C_006,2.099579e-02,450,650,2.890313,0.004,2.981866e-02,True,0.196000,False
101,AMG_2023-10-03_4_Channel.C_025_Unit 2,2.130100e-02,1500,1650,2.033622,0.046,2.981866e-02,True,0.750978,False
109,AMG_2023-10-04_1_Channel.C_004_Unit 1,1.522773e-14,150,400,6.857850,0.000,6.217995e-14,True,0.000000,True
116,AMG_2023-10-04_1_Channel.C_019_Unit 3,1.124395e-21,150,400,4.967133,0.000,5.366558e-21,True,0.000000,True
127,AMG_2023-10-04_2_Channel.C_018_Unit 1,2.978656e-05,150,500,12.549542,0.000,7.201685e-05,True,0.000000,True
133,AMG_2023-10-04_3_Channel.C_002_Unit 1,9.487733e-05,150,400,5.019932,0.001,2.158973e-04,True,0.066818,False
134,AMG_2023-10-04_3_Channel.C_004_Unit 2,7.482052e-157,100,500,2.210789,0.035,1.718534e-155,True,0.716154,False


In [96]:
from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader
ed_list = pd.read_csv('/old/Ed and ANOVA/used_for_R01/zombies_all_anova_passed_cells--used for grant.csv')
ed_list

,Unnamed: 0,Date,Round No.,Time Window,Cell,P Value
0,0,2023-09-26,1,"(0.0, 300.0)",Channel.C_027_Unit 1,0.030966
1,1,2023-09-26,2,"(700.0, 1000.0)",Channel.C_011_Unit 1,0.022412
2,2,2023-09-26,2,"(50.0, 150.0)",Channel.C_020,0.004885
3,3,2023-09-26,3,"(0.0, 2000.0)",Channel.C_002_Unit 1,0.000105
4,4,2023-09-26,3,"(100.0, 200.0)",Channel.C_007_Unit 2,0.031938
...,...,...,...,...,...,...
58,70,2023-12-14,4,"(1900.0, 2000.0)",Channel.C_006,0.014833
59,71,2023-12-14,4,"(750.0, 850.0)",Channel.C_008,0.042302
60,72,2023-12-18,2,"(200.0, 350.0)",Channel.C_012_Unit 2,0.037932
61,73,2023-12-18,3,"(2000.0, 2150.0)",Channel.C_015_Unit 1,0.037427


In [97]:
reader = RecordingMetadataReader()
prelim = reader.get_metadata_for_preliminary_analysis()
prelim['Date'] = prelim['Date'].astype(str)
ed_list = ed_list.merge(
    prelim[['Date', 'Round No.', 'Location']],
    on=['Date', 'Round No.'],
    how='left'
)
ed_list['NeuronID'] = (
        ed_list['Location'].astype(str) + "_" +
        ed_list['Date'].astype(str) + "_" +
        ed_list['Round No.'].astype(str) + "_" +
        ed_list['Cell'].astype(str)
)
# Step 1: Remove parentheses and split
ed_list[['WindowStart_ms', 'WindowEnd_ms']] = (
    ed_list['Time Window']
    .str.strip('()')               # Remove '(' and ')'
    .str.split(',', expand=True)  # Split into two columns
    .astype(float)                  # Convert to integers
)
ed_list.to_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/Ed and ANOVA/used_for_R01/ed_list_duplicate_window_removed.pkl')

In [98]:

is_full_window = (
    (ed_list['WindowStart_ms'] == 0) &
    (ed_list['WindowEnd_ms'] == 2000)
)

ed_list_without_full_window = ed_list[~is_full_window]
ed_list_without_full_window

,Unnamed: 0,Date,Round No.,Time Window,Cell,P Value,Location,NeuronID,WindowStart_ms,WindowEnd_ms
0,0,2023-09-26,1,"(0.0, 300.0)",Channel.C_027_Unit 1,3.096572e-02,AMG,AMG_2023-09-26_1_Channel.C_027_Unit 1,0.0,300.0
1,1,2023-09-26,2,"(700.0, 1000.0)",Channel.C_011_Unit 1,2.241222e-02,AMG,AMG_2023-09-26_2_Channel.C_011_Unit 1,700.0,1000.0
2,2,2023-09-26,2,"(50.0, 150.0)",Channel.C_020,4.884677e-03,AMG,AMG_2023-09-26_2_Channel.C_020,50.0,150.0
4,4,2023-09-26,3,"(100.0, 200.0)",Channel.C_007_Unit 2,3.193847e-02,AMG,AMG_2023-09-26_3_Channel.C_007_Unit 2,100.0,200.0
5,5,2023-09-26,3,"(150.0, 350.0)",Channel.C_027_Unit 2,4.591343e-02,AMG,AMG_2023-09-26_3_Channel.C_027_Unit 2,150.0,350.0
6,6,2023-10-03,3,"(200.0, 400.0)",Channel.C_006_Unit 1,2.052992e-04,AMG,AMG_2023-10-03_3_Channel.C_006_Unit 1,200.0,400.0
7,7,2023-10-03,3,"(2150.0, 2250.0)",Channel.C_006_Unit 1,2.199031e-04,AMG,AMG_2023-10-03_3_Channel.C_006_Unit 1,2150.0,2250.0
8,8,2023-10-03,3,"(200.0, 600.0)",Channel.C_013_Unit 1,3.527855e-02,AMG,AMG_2023-10-03_3_Channel.C_013_Unit 1,200.0,600.0
9,9,2023-10-03,3,"(1900.0, 2000.0)",Channel.C_026_Unit 1,1.277254e-03,AMG,AMG_2023-10-03_3_Channel.C_026_Unit 1,1900.0,2000.0
10,10,2023-10-03,4,"(450.0, 650.0)",Channel.C_006,7.075583e-03,AMG,AMG_2023-10-03_4_Channel.C_006,450.0,650.0


In [99]:
to_be_saved = ed_list_without_full_window[['NeuronID','WindowStart_ms', 'WindowEnd_ms', 'P Value']]
to_be_saved.to_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/Ed and ANOVA/used_for_R01/R01_ed_list_duplicate_removed_without_full_window.pkl')

In [90]:
to_be_saved = ed_list_without_full_window[['NeuronID','WindowStart_ms', 'WindowEnd_ms', 'P Value']]
to_be_saved.to_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/Ed and ANOVA/used_for_R01/R01_ed_list_without_full_window.pkl')

In [80]:
# Assume df1 and df2 are your input DataFrames
# Keep only relevant columns
df1_filtered = ed_list_without_full_window[['NeuronID', 'WindowStart_ms', 'WindowEnd_ms']].copy()
df2_filtered = my_list[['NeuronID', 'WindowStart_ms', 'WindowEnd_ms']].copy()

# Rename for clarity
df1_filtered.rename(columns={'WindowStart_ms': 'WindowStart_df1', 'WindowEnd_ms': 'WindowEnd_df1'}, inplace=True)
df2_filtered.rename(columns={'WindowStart_ms': 'WindowStart_df2', 'WindowEnd_ms': 'WindowEnd_df2'}, inplace=True)

# Merge on NeuronID
merged = pd.merge(df1_filtered, df2_filtered, on='NeuronID', how='outer')

# Compare window match
merged['WindowMatch'] = (
    (merged['WindowStart_df1'] == merged['WindowStart_df2']) &
    (merged['WindowEnd_df1'] == merged['WindowEnd_df2'])
)

overlap_neurons = list(set(df1_filtered['NeuronID']) & set(df2_filtered['NeuronID']))

# NeuronIDs only in df1
only_in_df1 = list(set(df1_filtered['NeuronID']) - set(df2_filtered['NeuronID']))
# NeuronIDs only in df2
only_in_df2 = list(set(df2_filtered['NeuronID']) - set(df1_filtered['NeuronID']))

# Show summary
print("NeuronIDs only in ed_list:", sorted(only_in_df1))
print(len(only_in_df1))

NeuronIDs only in ed_list: ['AMG_2023-09-26_1_Channel.C_027_Unit 1', 'AMG_2023-09-26_2_Channel.C_011_Unit 1', 'AMG_2023-09-26_2_Channel.C_020', 'AMG_2023-09-26_3_Channel.C_007_Unit 2', 'AMG_2023-10-03_3_Channel.C_013_Unit 1', 'AMG_2023-10-03_3_Channel.C_026_Unit 1', 'AMG_2023-10-03_4_Channel.C_010_Unit 2', 'AMG_2023-10-03_4_Channel.C_026', 'AMG_2023-10-05_1_Channel.C_004_Unit 1', 'AMG_2023-10-05_2_Channel.C_009_Unit 1', 'AMG_2023-10-05_2_Channel.C_019_Unit 2', 'AMG_2023-10-11_1_Channel.C_002', 'AMG_2023-10-11_1_Channel.C_010', 'ER_2023-10-24_2_Channel.C_002', 'ER_2023-11-20_1_Channel.C_013', 'ER_2023-11-20_2_Channel.C_006', 'ER_2023-11-25_3_Channel.C_018', 'ER_2023-12-18_2_Channel.C_012_Unit 2', 'ER_2023-12-18_3_Channel.C_015_Unit 1', 'ER_2023-12-18_3_Channel.C_020_Unit 1']
20


In [81]:
print("NeuronIDs only in my_list:", sorted(only_in_df2))
print(len(only_in_df2))

NeuronIDs only in my_list: ['AMG_2023-10-03_4_Channel.C_025_Unit 2', 'ER_2023-11-22_2_Channel.C_027', 'Unknown_2023-10-31_1_Channel.C_021', 'Unknown_2023-11-08_1_Channel.C_005', 'Unknown_2023-11-08_2_Channel.C_016', 'Unknown_2023-12-18_2_Channel.C_017_Unit 1', 'Unknown_2023-12-18_3_Channel.C_015_Unit 1']
7


In [83]:
print("NeuronIDs in both my_list and ed's list :")
print(len(overlap_neurons))

NeuronIDs in both my_list and ed's list :
35


In [38]:
merged

,NeuronID,WindowStart_df1,WindowEnd_df1,WindowStart_df2,WindowEnd_df2,WindowMatch
0,AMG_2023-09-26_1_Channel.C_027_Unit 1,0.0,300.0,NaN,NaN,False
1,AMG_2023-09-26_2_Channel.C_011_Unit 1,700.0,1000.0,NaN,NaN,False
2,AMG_2023-09-26_2_Channel.C_020,50.0,150.0,NaN,NaN,False
3,AMG_2023-09-26_3_Channel.C_007_Unit 2,100.0,200.0,NaN,NaN,False
4,AMG_2023-09-26_3_Channel.C_027_Unit 2,150.0,350.0,150.0,350.0,True
...,...,...,...,...,...,...
75,Unknown_2023-10-31_1_Channel.C_021,NaN,NaN,2550.0,2700.0,False
76,Unknown_2023-11-08_1_Channel.C_005,NaN,NaN,1100.0,1200.0,False
77,Unknown_2023-11-08_2_Channel.C_016,NaN,NaN,350.0,450.0,False
78,Unknown_2023-12-18_2_Channel.C_017_Unit 1,NaN,NaN,800.0,900.0,False


In [41]:
merged[merged['WindowStart_df1'].isna()]

,NeuronID,WindowStart_df1,WindowEnd_df1,WindowStart_df2,WindowEnd_df2,WindowMatch
73,AMG_2023-10-03_4_Channel.C_025_Unit 2,NaN,NaN,1500.0,1650.0,False
74,ER_2023-11-22_2_Channel.C_027,NaN,NaN,200.0,450.0,False
75,Unknown_2023-10-31_1_Channel.C_021,NaN,NaN,2550.0,2700.0,False
76,Unknown_2023-11-08_1_Channel.C_005,NaN,NaN,1100.0,1200.0,False
77,Unknown_2023-11-08_2_Channel.C_016,NaN,NaN,350.0,450.0,False
78,Unknown_2023-12-18_2_Channel.C_017_Unit 1,NaN,NaN,800.0,900.0,False
79,Unknown_2023-12-18_3_Channel.C_015_Unit 1,NaN,NaN,2000.0,2150.0,False


In [45]:
# Convert to set if not already
only_in_df1_set = set(only_in_df1)
neuron_ids_df3 = set(windows['NeuronID'])

# Neurons in only_in_df1 that are also in df3
intersect = only_in_df1_set & neuron_ids_df3

# Neurons in only_in_df1 that are NOT in df3
missing = only_in_df1_set - neuron_ids_df3

# Report
print(f"✅ {len(intersect)} neurons from only in ed's list are detected by original window finder algorithm.")
print(f"❌ {len(missing)} neurons from only in ed's list are NOT detected by original window finder algorithm..")
missing

✅ 7 neurons from only in ed's list are detected by original window finder algorithm.
❌ 13 neurons from only in ed's list are NOT detected by original window finder algorithm..


{'AMG_2023-09-26_2_Channel.C_011_Unit 1',
 'AMG_2023-09-26_3_Channel.C_007_Unit 2',
 'AMG_2023-10-03_3_Channel.C_026_Unit 1',
 'AMG_2023-10-03_4_Channel.C_010_Unit 2',
 'AMG_2023-10-03_4_Channel.C_026',
 'AMG_2023-10-05_1_Channel.C_004_Unit 1',
 'AMG_2023-10-05_2_Channel.C_019_Unit 2',
 'ER_2023-11-20_1_Channel.C_013',
 'ER_2023-11-20_2_Channel.C_006',
 'ER_2023-11-25_3_Channel.C_018',
 'ER_2023-12-18_2_Channel.C_012_Unit 2',
 'ER_2023-12-18_3_Channel.C_015_Unit 1',
 'ER_2023-12-18_3_Channel.C_020_Unit 1'}

In [46]:
# Step 1: Subset only_in_df1 neurons
only_df1_subset = df1_filtered[df1_filtered['NeuronID'].isin(only_in_df1)]

# Step 2: Identify which have full [0, 2000] window
is_full_window = (
    (only_df1_subset['WindowStart_df1'] == 0) &
    (only_df1_subset['WindowEnd_df1'] == 2000)
)

# Step 3: Get NeuronIDs that are NOT full window (i.e., potentially meaningful)
only_in_df1_useful = only_df1_subset[~is_full_window]['NeuronID']

# Step 4: Compare with df3 (e.g., `windows`)
only_in_df1_set = set(only_in_df1_useful)
neuron_ids_df3 = set(windows['NeuronID'])

# Step 5: Check overlap
intersect = only_in_df1_set & neuron_ids_df3
missing = only_in_df1_set - neuron_ids_df3

# Step 6: Report
print(f"✅ {len(intersect)} neurons from 'only in ed's list' (excluding [0, 2000] ones) are detected by original window finder algorithm.")
print(f"❌ {len(missing)} neurons from 'only in ed's list' (excluding [0, 2000] ones) are NOT detected by original window finder algorithm.")

✅ 7 neurons from 'only in ed's list' (excluding [0, 2000] ones) are detected by original window finder algorithm.
❌ 13 neurons from 'only in ed's list' (excluding [0, 2000] ones) are NOT detected by original window finder algorithm.


In [47]:
missing

{'AMG_2023-09-26_2_Channel.C_011_Unit 1',
 'AMG_2023-09-26_3_Channel.C_007_Unit 2',
 'AMG_2023-10-03_3_Channel.C_026_Unit 1',
 'AMG_2023-10-03_4_Channel.C_010_Unit 2',
 'AMG_2023-10-03_4_Channel.C_026',
 'AMG_2023-10-05_1_Channel.C_004_Unit 1',
 'AMG_2023-10-05_2_Channel.C_019_Unit 2',
 'ER_2023-11-20_1_Channel.C_013',
 'ER_2023-11-20_2_Channel.C_006',
 'ER_2023-11-25_3_Channel.C_018',
 'ER_2023-12-18_2_Channel.C_012_Unit 2',
 'ER_2023-12-18_3_Channel.C_015_Unit 1',
 'ER_2023-12-18_3_Channel.C_020_Unit 1'}

In [49]:
ed_list_without_full_window[['NeuronID','WindowStart_ms', 'WindowEnd_ms']]

,NeuronID,WindowStart_ms,WindowEnd_ms
0,AMG_2023-09-26_1_Channel.C_027_Unit 1,0.0,300.0
1,AMG_2023-09-26_2_Channel.C_011_Unit 1,700.0,1000.0
2,AMG_2023-09-26_2_Channel.C_020,50.0,150.0
4,AMG_2023-09-26_3_Channel.C_007_Unit 2,100.0,200.0
5,AMG_2023-09-26_3_Channel.C_027_Unit 2,150.0,350.0
...,...,...,...
70,ER_2023-12-14_4_Channel.C_006,1900.0,2000.0
71,ER_2023-12-14_4_Channel.C_008,750.0,850.0
72,ER_2023-12-18_2_Channel.C_012_Unit 2,200.0,350.0
73,ER_2023-12-18_3_Channel.C_015_Unit 1,2000.0,2150.0


In [50]:
ed_list[['NeuronID','WindowStart_ms', 'WindowEnd_ms']]

,NeuronID,WindowStart_ms,WindowEnd_ms
0,AMG_2023-09-26_1_Channel.C_027_Unit 1,0.0,300.0
1,AMG_2023-09-26_2_Channel.C_011_Unit 1,700.0,1000.0
2,AMG_2023-09-26_2_Channel.C_020,50.0,150.0
3,AMG_2023-09-26_3_Channel.C_002_Unit 1,0.0,2000.0
4,AMG_2023-09-26_3_Channel.C_007_Unit 2,100.0,200.0
...,...,...,...
70,ER_2023-12-14_4_Channel.C_006,1900.0,2000.0
71,ER_2023-12-14_4_Channel.C_008,750.0,850.0
72,ER_2023-12-18_2_Channel.C_012_Unit 2,200.0,350.0
73,ER_2023-12-18_3_Channel.C_015_Unit 1,2000.0,2150.0


In [59]:
overlap_list=[]
for _,row in ed_list[is_full_window].iterrows():
    neuron = row['NeuronID']
    overlap = ed_list_without_full_window[ed_list_without_full_window['NeuronID']==neuron]
    overlap_list.append(overlap)

final_df = pd.concat(overlap_list, ignore_index=True)
final_df

,Date,Round No.,Time Window,Cell,P Value,Location,NeuronID,WindowStart_ms,WindowEnd_ms
0,2023-10-04,3,"(150.0, 500.0)",Channel.C_009_Unit 1,0.001049,AMG,AMG_2023-10-04_3_Channel.C_009_Unit 1,150.0,500.0
1,2023-10-11,3,"(1650.0, 1800.0)",Channel.C_013,0.005806,AMG,AMG_2023-10-11_3_Channel.C_013,1650.0,1800.0


In [66]:
windows['NeuronID'].nunique()

394

In [1]:
import pandas as pd
dat = pd.read_pickle('/home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache/2023-09-26_round_3.pkl')
dat

,TaskField,MonkeyId,MonkeyGroup,MonkeyName,Channel,SpikeTimes,EpochStartStop,BaseChannel,Date,Round No.,Location,NeuronID
0,1695753698845000,2854,Stranger Things,40J,Channel.C_012_Unit 1,[],"(3.9929, 6.5124)",Channel.C_012,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_012_Unit 1
1,1695753698845000,2854,Stranger Things,40J,Channel.C_012_Unit 2,[],"(3.9929, 6.5124)",Channel.C_012,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_012_Unit 2
2,1695753698845000,2854,Stranger Things,40J,Channel.C_027_Unit 1,"[4.4698, 4.4734, 4.6099, 4.61325, 4.62355, 4.6...","(3.9929, 6.5124)",Channel.C_027,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_027_Unit 1
3,1695753698845000,2854,Stranger Things,40J,Channel.C_027_Unit 2,"[3.99515, 4.0247, 4.05085, 4.05665, 4.08425, 4...","(3.9929, 6.5124)",Channel.C_027,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_027_Unit 2
4,1695753698845000,2854,Stranger Things,40J,Channel.C_027_Unit 3,"[4.0088, 4.01415, 4.02375, 4.03355, 4.04005, 4...","(3.9929, 6.5124)",Channel.C_027,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_027_Unit 3
...,...,...,...,...,...,...,...,...,...,...,...,...
10803,1695753725090000,1723,Instigators,42Z,Channel.C_004,"[8.32005, 8.3231, 8.3273, 8.6976, 8.70055, 8.7...","(8.1331, 10.70825)",Channel.C_004,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_004
10804,1695753725090000,1723,Instigators,42Z,Channel.C_005,[],"(8.1331, 10.70825)",Channel.C_005,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_005
10805,1695753725090000,1723,Instigators,42Z,Channel.C_006,"[8.17455, 8.1865, 8.9367]","(8.1331, 10.70825)",Channel.C_006,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_006
10808,1695753725090000,1723,Instigators,42Z,Channel.C_013,"[8.16855, 8.17335, 8.26025, 8.2657, 8.41085, 8...","(8.1331, 10.70825)",Channel.C_013,2023-09-26,3,AMG,AMG_2023-09-26_3_Channel.C_013
